In [2]:
import numpy as np
from keras.src.datasets.mnist import load_data as mnist

In [4]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print(X_train)

[[[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 ...

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]

 [[0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  ...
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]
  [0 0 0 ... 0 0 0]]]


In [7]:
X_train = X_train[:1000]
labels = y_train[:1000]
X_train = X_train.reshape(1000, 28*28)
images = X_train / 255

print(images)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [6]:
OHV = np.zeros((len(labels), 10))
for idx, label in enumerate(labels):
  OHV[idx][label] = 1
labels1 = OHV

print(labels1)

[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [10]:
images1 = X_test.reshape((len(X_test), 28*28))
x_test = images1 / 255
OHV1 = np.zeros((len(y_test), 10))
for idx, label in enumerate(y_test):
  OHV1[idx][label] = 1
test_labels = OHV1

print(test_labels)

[[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [ ]:
alpha, iterations = (2, 1)
pixels_per_image = (784)
batch_size = 128
input_rows = 28
input_columns = 28
kernel_rows = 3
kernel_columns = 3
num_kernels = 16
num_labels = 10

hidden_size = ((input_rows - kernel_rows) * (input_columns - kernel_columns)) * num_kernels

kernels = np.random.random((kernel_rows * kernel_columns, num_kernels))

weights_1_2 = np.random.random((hidden_size, num_labels))

def get_image_section(layer, row_from, row_to, col_from, col_to):
  section = layer[:, row_from:row_to, col_from:col_to]
  reshaped_section = section.reshape(-1, 1, row_to - row_from, col_to - col_from)
  return reshaped_section

def tanh(x):
  return np.tanh(x)

def tanh2deriv(output):
  return 1-(output**2)

def softmax(x):
  temp = np.exp(x)
  return temp / (np.sum(temp, axis=1, keepdims=True))

for j in range(iterations):
  correct_cnt = 0
  for i in range(int(len(images) / batch_size)):
    batch_start, batch_end = ((i * batch_size), ((i+1) * batch_size))
    layer_0 = images[batch_start:batch_end]
    layer_0 = layer_0.reshape(layer_0.shape[0], 28, 28)
    sects = []
    for row_start in range(layer_0.shape[1] - kernel_rows):
      for col_start in range(layer_0.shape[2] - kernel_columns):
        sect = get_image_section(layer_0, row_start, row_start + kernel_rows, col_start, col_start + kernel_columns)
        sects.append(sect)
    expanded_input = np.concatenate(sects, axis=1)
    es = expanded_input.shape
    flattened_input = expanded_input.reshape(es[0] * es[1], -1)
    kernel_output = flattened_input.dot(kernels)
    layer_1 = tanh(kernel_output.reshape(es[0], -1))
    dropout_mask = np.random.randint(2, size=layer_1.shape)
    layer_1 *= dropout_mask * 2
    layer_2 = softmax(np.dot(layer_1, weights_1_2))
    layer_2_delta = (labels1[batch_start:batch_end] - layer_2) / batch_size
    layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
    layer_1_delta *= dropout_mask
    weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
    l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
    k_update = flattened_input.T.dot(l1d_reshape)
    kernels -= alpha * k_update
